In [ ]:
import math
import requests
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from skyfield.api import load, EarthSatellite, wgs84
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, roc_auc_score , confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.calibration import CalibratedClassifierCV
import pulp
import matplotlib.pyplot as plt

# ---------------------- Configurações ----------------------
STATION = {
    'name': 'São Paulo, Brazil',
    'lat': -23.2156,
    'lon': -45.8859,
    'alt_m': 650
}




HISTORY_DAYS = 80
FUTURE_DAYS = 10
CELESTRAK_CATEGORY = 'active'
SAT_NAME_CONTAINS = "Starlink"#'CALSPHERE'

PRECIP_THRESHOLD_MM = 0.5
ELEVATION_MIN_DEG = 20.0

RANDOM_STATE = 42
BITRATE_KBPS = 1000.0

# Novos parâmetros:
LABEL_FLIP_PROB = 0.05   # probabilidade de inverter o rótulo (simula ruído/imperfeição)
MIN_SETUP_S = 60*5         # tempo mínimo de setup entre passes (segundos) — torna conflito mais realista
MAX_SCHEDULED = None     # limite máximo de passes agendados (None = sem limite)
 
SETUP_COST_MULTIPLIER = 0.3
SETUP_COST_KB = (MIN_SETUP_S * BITRATE_KBPS) / 1000.0 * SETUP_COST_MULTIPLIER
MAX_TOTAL_SCHEDULED_S = 7200          # 1 hora de janelas agendáveis                # ou 3, 5, etc. para testar limite 

 

'''# novo parâmetro
SETUP_COST_MULTIPLIER = 0.5  # multiplica o custo mínimo (ajuste)
# custo (kB) equivalente ao tempo de setup a ser penalizado, por exemplo:
SETUP_COST_KB = (MIN_SETUP_S * BITRATE_KBPS) / 1000.0 * SETUP_COST_MULTIPLIER
'''

rng = np.random.RandomState(RANDOM_STATE)

# ---------------------- Utilitários ----------------------
def fetch_tle_from_celestrak(category=CELESTRAK_CATEGORY):
    url = f'https://celestrak.org/NORAD/elements/gp.php?GROUP={category}&FORMAT=tle'
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return r.text

def tle_to_dataframe(tle_text):
    # Divide o texto em linhas
    lines = [line.strip() for line in tle_text.strip().split("\n") if line.strip()]
    
    # Cada TLE tem 3 linhas: nome, linha 1, linha 2
    records = []
    for i in range(0, len(lines), 3):
        try:
            name = lines[i]
            line1 = lines[i+1]
            line2 = lines[i+2]
            records.append((name, line1, line2))
        except IndexError:
            # Caso o último registro esteja incompleto
            pass
    
    df = pd.DataFrame(records, columns=["Satellite", "Line1", "Line2"])
    return df

def save_tle_as_csv(category="active", filename="tle_data.csv"):
    tle_text = fetch_tle_from_celestrak(category)
    df = tle_to_dataframe(tle_text)
    df.to_csv(filename, index=False)
    print(f"✅ Dataset salvo com sucesso em: {filename}")

# Exemplo de uso:
save_tle_as_csv("active", "tle_active.csv")

def select_satellite_tle(tle_text, name_contains=SAT_NAME_CONTAINS):
    lines = [l.strip() for l in tle_text.splitlines() if l.strip()]
    blocks = [lines[i:i+3] for i in range(0, len(lines), 3)]
    for block in blocks:
        if len(block) >= 3 and name_contains.upper() in block[0].upper():
            return block[0], block[1], block[2]
    return blocks[0][0], blocks[0][1], blocks[0][2]

def compute_passes_for_history(sat, station, history_days=HISTORY_DAYS, min_elevation_deg=20.0):
    ts = load.timescale()
    end = datetime.utcnow()
    start = end - timedelta(days=history_days)

    step_minutes = 1
    times = []
    cur = start
    while cur <= end:
        times.append(cur)
        cur += timedelta(minutes=step_minutes)
    t_ts = ts.utc([t.year for t in times], [t.month for t in times], [t.day for t in times],
                  [t.hour for t in times], [t.minute for t in times], [t.second for t in times])

    station_point = wgs84.latlon(station['lat'], station['lon'], station['alt_m'])
    difference = sat - station_point
    topocentric = difference.at(t_ts)
    alt, az, distance = topocentric.altaz()
    elevations_deg = alt.degrees

    is_visible = elevations_deg > 0
    passes = []
    i = 0
    pass_idx = 0
    while i < len(is_visible):
        if is_visible[i]:
            j = i
            while j + 1 < len(is_visible) and is_visible[j + 1]:
                j += 1
            pass_times = times[i:j+1]
            pass_elevs = elevations_deg[i:j+1]
            max_elev = float(np.max(pass_elevs))
            duration_s = (pass_times[-1] - pass_times[0]).total_seconds()
            if max_elev >= min_elevation_deg and duration_s >= 10:
                passes.append({
                    'pass_id': f'pass_{pass_idx}',
                    'start_utc': pass_times[0],
                    'end_utc': pass_times[-1],
                    'duration_s': duration_s,
                    'max_elevation_deg': max_elev,
                    'sample_times': pass_times,
                    'elevations_deg': pass_elevs
                })
                pass_idx += 1
            i = j + 1
        else:
            i += 1

    df = pd.DataFrame(passes)
    return df


def fetch_open_meteo_hourly(lat, lon, start_dt, end_dt):
    ''' buscar dados de previsão do tempo de um serviço online gratuito chamado Open-Meteo'''
    start_iso = start_dt.strftime('%Y-%m-%dT%H:%M')
    end_iso = end_dt.strftime('%Y-%m-%dT%H:%M')
    url = (
        'https://api.open-meteo.com/v1/forecast'
        f'?latitude={lat}&longitude={lon}'
        '&hourly=precipitation,cloudcover'
        f'&start_date={start_dt.date()}&end_date={end_dt.date()}'
        '&timezone=UTC'
    )
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    j = r.json()
    hours = j.get('hourly', {}).get('time', [])
    precip = j.get('hourly', {}).get('precipitation', [])
    cloud = j.get('hourly', {}).get('cloudcover', [])
    df = pd.DataFrame({'time': hours, 'precipitation': precip, 'cloudcover': cloud})
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    return df

def features_from_pass(pass_row, meteo_df):

    
    start = pd.to_datetime(pass_row['start_utc'])
    end = pd.to_datetime(pass_row['end_utc'])
    sel = meteo_df.loc[(meteo_df.index >= start) & (meteo_df.index <= end)]
    if sel.empty:
        precip_mean = 0.0
        cloud_mean = 0.0
        precip_max = 0.0
        cloud_max = 0.0
        precip_count = 0
    else:
        precip_mean = float(sel['precipitation'].mean())
        cloud_mean = float(sel['cloudcover'].mean())
        precip_max = float(sel['precipitation'].max())
        cloud_max = float(sel['cloudcover'].max())
        precip_count = int((sel['precipitation'] > 0.1).sum())

    elevs = np.array(pass_row.get('elevations_deg', []), dtype=float)
    if elevs.size == 0:
        elev_mean = pass_row['max_elevation_deg']
        elev_std = 0.0
    else:
        elev_mean = float(np.mean(elevs))
        elev_std = float(np.std(elevs))

    # cyclic time features (UTC midpoint)
    midpoint = start + (end - start) / 2
    hour = midpoint.hour + midpoint.minute/60.0
    hour_sin = math.sin(2*math.pi*hour/24.0)
    hour_cos = math.cos(2*math.pi*hour/24.0)

    day_of_year = midpoint.timetuple().tm_yday
    doy_sin = math.sin(2*math.pi*day_of_year/365.25)
    doy_cos = math.cos(2*math.pi*day_of_year/365.25)

    return {
        'duration_s': pass_row['duration_s'],
        'max_elevation_deg': pass_row['max_elevation_deg'],
        'elevation_mean_deg': elev_mean,
        'elevation_std_deg': elev_std,
        'precipitation_mean': precip_mean,
        'precipitation_max': precip_max,
        'precip_hours_count': precip_count,
        'cloudcover_mean': cloud_mean,
        'cloudcover_max': cloud_max,
        'hour_sin': hour_sin,
        'hour_cos': hour_cos,
        'doy_sin': doy_sin,
        'doy_cos': doy_cos
    }

def label_pass_proxy(features_row, precip_threshold=PRECIP_THRESHOLD_MM, elev_threshold=ELEVATION_MIN_DEG, flip_prob=LABEL_FLIP_PROB):
    # regra base (proxy) — depois aplicamos flip randômico para simular rótulos ruidosos
    success = 1
    if (features_row['precipitation_mean'] > precip_threshold) or (features_row['max_elevation_deg'] < elev_threshold):
        success = 0
    # flip com probabilidade pequena para introduzir incerteza
    if rng.rand() < flip_prob:
        success = 1 - success
    return int(success)

def build_dataset(sat, station, history_days=HISTORY_DAYS):
    print('Computando passes históricos...')
    passes_df = compute_passes_for_history(sat, station, history_days=history_days)
    print(f'Encontrados {len(passes_df)} passes no histórico')

    if passes_df.empty:
        return pd.DataFrame()
    global_start = passes_df['start_utc'].min()
    global_end = passes_df['end_utc'].max()
    meteo = fetch_open_meteo_hourly(station['lat'], station['lon'], global_start - timedelta(hours=1), global_end + timedelta(hours=1))

    records = []
    for _, row in passes_df.iterrows():
        feats = features_from_pass(row, meteo)
        label = label_pass_proxy(feats)
        rec = {
            'pass_id': row['pass_id'],
            'start_utc': row['start_utc'],
            'end_utc': row['end_utc'],
            **feats,
            'label_success': label
        }
        records.append(rec)
    return pd.DataFrame(records)

# ---------------------- Treino e calibração ----------------------
def train_model(df):
    feature_cols = [c for c in df.columns if c not in ['pass_id','start_utc','end_utc','label_success']]
    X = df[feature_cols].values
    y = df['label_success'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    base_clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
    # CalibratedClassifierCV com cv stratificado
    calib = CalibratedClassifierCV(base_clf, cv=5, method='sigmoid')
    calib.fit(X_train, y_train)

    # avaliação cruzada (probabilidades por CV) para estimar AUC robusto
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    probas_cv = cross_val_predict(calib, X, y, cv=skf, method='predict_proba')[:, 1]
   
    auc_cv = roc_auc_score(y, probas_cv)
    print(f'CV AUC (calibrated): {auc_cv:.3f}')
   

    # relatório simples no holdout (split)    
    y_pred = calib.predict(X_test)
    y_proba = calib.predict_proba(X_test)[:, 1]
    print('Classification report (test):')
    print(classification_report(y_test, y_pred))
   
    auc = roc_auc_score(y_test, y_proba)
    print(f'AUC (test): {auc:.3f}')


    f1 = f1_score(y_test, y_pred)
    print(f'F1 Score: {f1:.3f}')    
    

    # Gera a matriz de confusão
    cm = confusion_matrix(y_test, y_pred)
    print("Matriz de confusão:")
    print(cm)

    # (Opcional) plota a matriz de confusão de forma visual
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.show()


    return calib, feature_cols

# ---------------------- Otimização MILP ----------------------
def schedule_milp(passes_df, success_probs, bitrate_kbps=BITRATE_KBPS,
                  min_setup_s=MIN_SETUP_S, max_selected=MAX_SCHEDULED,
                  setup_cost_kb=0.0, max_total_scheduled_s=None):
    """
    - min_setup_s: tempo mínimo de setup (segundos) adicionado antes do start e depois do end para checar conflito
    - setup_cost_kb: penalidade fixa (kB) por cada pass agendado (faz solver evitar muitos passes pequenos)
    - max_total_scheduled_s: limite em segundos do tempo total agendado (capacidade)
    """
    prob = pulp.LpProblem('schedule', pulp.LpMaximize)
    n = len(passes_df)
    x = [pulp.LpVariable(f'x_{i}', cat='Binary') for i in range(n)]

    values = [float(passes_df.iloc[i]['duration_s']) * float(success_probs[i]) * bitrate_kbps for i in range(n)]
    # objetivo = soma(values * x) - setup_cost_kb * sum(x)
    prob += pulp.lpSum([values[i] * x[i] for i in range(n)]) - setup_cost_kb * pulp.lpSum(x)

    # Restrições de sobreposição considerando setup antes e depois
    starts = pd.to_datetime(passes_df['start_utc'])
    ends = pd.to_datetime(passes_df['end_utc'])
    setup_td = pd.Timedelta(seconds=int(min_setup_s))
    for i in range(n):
        si = starts.iloc[i] - setup_td
        ei = ends.iloc[i] + setup_td
        for j in range(i+1, n):
            sj = starts.iloc[j] - setup_td
            ej = ends.iloc[j] + setup_td
            # conflito se os intervalos expandidos se sobrepõem
            if (si < ej) and (sj < ei):
                prob += x[i] + x[j] <= 1

    # capacidade de tempo total (opcional)
    if max_total_scheduled_s is not None:
        prob += pulp.lpSum([passes_df.iloc[i]['duration_s'] * x[i] for i in range(n)]) <= float(max_total_scheduled_s)

    # limite de número de agendamentos (opcional)
    if max_selected is not None:
        prob += pulp.lpSum(x) <= int(max_selected)

    status = prob.solve(pulp.PULP_CBC_CMD(msg=False))
    if pulp.LpStatus[status] != 'Optimal':
        print(f"[Aviso] Solver retornou status: {pulp.LpStatus[status]}")

    chosen = []
    for i in range(n):
        val = pulp.value(x[i])
        chosen.append(bool(val is not None and val > 0.5))
    return chosen

# ---------------------- Baselines ----------------------
def greedy_schedule_by_start(passes_df):
    df = passes_df.copy().sort_values('end_utc')
    chosen = []
    current_end = None
    for idx, row in df.iterrows():
        if (current_end is None) or (pd.to_datetime(row['start_utc']) >= current_end):
            chosen.append(row['pass_id'])
            current_end = pd.to_datetime(row['end_utc'])
    return set(chosen)


def greedy_schedule_by_expected_value(passes_df, expected_values, min_setup_s=MIN_SETUP_S, setup_cost_kb=0.0):
    order = np.argsort(-np.array(expected_values))
    chosen = set()
    schedule = []
    for idx in order:
        row = passes_df.iloc[idx]
        s = pd.to_datetime(row['start_utc']) - pd.Timedelta(seconds=int(min_setup_s))
        e = pd.to_datetime(row['end_utc']) + pd.Timedelta(seconds=int(min_setup_s))
        value = expected_values[idx] - setup_cost_kb*1000.0  # expected_values em [bits*seg]? ajuste conforme unidade
        # (melhore aqui a unidade: se expected_values estiver em kB já, subtraia setup_cost_kb direto)
        conflict = False
        for c in schedule:
            cs = pd.to_datetime(c['start_utc']) - pd.Timedelta(seconds=int(min_setup_s))
            ce = pd.to_datetime(c['end_utc']) + pd.Timedelta(seconds=int(min_setup_s))
            if (s < ce) and (cs < e):
                conflict = True
                break
        if not conflict and value > 0:
            schedule.append({'pass_id': row['pass_id'], 'start_utc': row['start_utc'], 'end_utc': row['end_utc']})
            chosen.add(row['pass_id'])
    return chosen

# Modificar a função evaluate_schedule para verificar conflitos
def evaluate_schedule(passes_df, selected_pass_ids, true_labels, pred_probs, bitrate_kbps=BITRATE_KBPS, min_setup_s=MIN_SETUP_S):
    # Verificar conflitos primeiro
    selected_indices = [i for i, row in passes_df.iterrows() if row['pass_id'] in selected_pass_ids]
    
    # Verificar conflitos entre passes selecionados
    starts = pd.to_datetime(passes_df['start_utc'])
    ends = pd.to_datetime(passes_df['end_utc'])
    setup_td = pd.Timedelta(seconds=int(min_setup_s))
    
    has_conflicts = False
    for i in selected_indices:
        for j in selected_indices:
            if i >= j:
                continue
            si = starts.iloc[i] - setup_td
            ei = ends.iloc[i] + setup_td
            sj = starts.iloc[j] - setup_td
            ej = ends.iloc[j] + setup_td
            if (si < ej) and (sj < ei):
                print(f"AVISO: Conflito entre passes {passes_df.iloc[i]['pass_id']} e {passes_df.iloc[j]['pass_id']}")
                has_conflicts = True
    
    if has_conflicts:
        print("AVISO: O agendamento contém conflitos!")
    
    # Cálculo normal de expected e realized
    idx_map = {passes_df.iloc[i]['pass_id']: i for i in range(len(passes_df))}
    realized = 0.0
    expected = 0.0
    for pid in selected_pass_ids:
        i = idx_map[pid]
        dur = passes_df.iloc[i]['duration_s']
        expected += dur * bitrate_kbps * float(pred_probs[i])
        realized += dur * bitrate_kbps * float(true_labels[i])
    return {'expected_kb': expected / 1000.0, 'realized_kb': realized / 1000.0, 'has_conflicts': has_conflicts}




In [ ]:

# ---------------------- Execução principal ----------------------
print('Baixando TLEs...')
tle_text = fetch_tle_from_celestrak()
name, l1, l2 = select_satellite_tle(tle_text)
print('Usando satélite:', name)

ts = load.timescale()
sat = EarthSatellite(l1, l2, name, ts)

df_hist = build_dataset(sat, STATION, history_days=HISTORY_DAYS)
if df_hist.empty:
    print('Nenhum pass encontrado no histórico. Ajuste HISTORY_DAYS ou localização da estação.')
else:
    clf_calib, feature_cols = train_model(df_hist)

    # Preparamos passes futuros
    print('Computando passes futuros para avaliação da política...')
    start_future = datetime.utcnow()
    end_future = start_future + timedelta(days=FUTURE_DAYS)
    step_minutes = 1
    times = []
    cur = start_future
    while cur <= end_future:
        times.append(cur)
        cur += timedelta(minutes=step_minutes)
    t_ts = ts.utc([t.year for t in times], [t.month for t in times], [t.day for t in times],
                    [t.hour for t in times], [t.minute for t in times], [t.second for t in times])

    difference = sat - wgs84.latlon(STATION['lat'], STATION['lon'], STATION['alt_m'])
    topocentric = difference.at(t_ts)
    alt, az, dist = topocentric.altaz()
    elevations_deg = alt.degrees
    is_visible = elevations_deg > 0

    passes = []
    i = 0
    pass_idx = 0
    while i < len(is_visible):
        if is_visible[i]:
            j = i
            while j + 1 < len(is_visible) and is_visible[j+1]:
                j += 1
            pass_times = times[i:j+1]
            pass_elevs = elevations_deg[i:j+1]
            max_elev = float(np.max(pass_elevs))
            duration_s = (pass_times[-1] - pass_times[0]).total_seconds()
            if max_elev >= 1.0 and duration_s >= 10:
                passes.append({
                    'pass_id': f'future_pass_{pass_idx}',
                    'start_utc': pass_times[0],
                    'end_utc': pass_times[-1],
                    'duration_s': duration_s,
                    'max_elevation_deg': max_elev,
                    'elevations_deg': pass_elevs
                })
                pass_idx += 1
            i = j+1
        else:
            i += 1

    df_future = pd.DataFrame(passes)
    print(f'Encontrados {len(df_future)} passes futuros para agendamento')
    if df_future.empty:
        print('Nenhum pass futuro encontrado. Ajuste FUTURE_DAYS.')
    else:
        meteo_future = fetch_open_meteo_hourly(STATION['lat'], STATION['lon'], start_future - timedelta(hours=1), end_future + timedelta(hours=1))
        X_future = []
        true_labels = []
        for _, row in df_future.iterrows():
            feats = features_from_pass(row, meteo_future)
            X_future.append([feats[c] for c in feature_cols])
            true_labels.append(label_pass_proxy(feats, flip_prob=0.0))  # para avaliação, usamos rótulo "verdadeiro" sem flip
            #true_labels.append(label_pass_proxy(feats, flip_prob=LABEL_FLIP_PROB))

        X_future = np.array(X_future)
        true_labels = np.array(true_labels)

        # Predict (calibrated)
        pred_probs = clf_calib.predict_proba(X_future)[:, 1]
        #print('Predicted probs:', np.round(pred_probs, 3))

        expected_values = [df_future.iloc[i]['duration_s'] * pred_probs[i] * BITRATE_KBPS for i in range(len(df_future))]
        #print('Expected values (kbps*duration*p):', expected_values)

        chosen_bool = schedule_milp(df_future, pred_probs, min_setup_s=MIN_SETUP_S,
                       setup_cost_kb=SETUP_COST_KB, max_total_scheduled_s=MAX_TOTAL_SCHEDULED_S)
        
        
        #schedule_milp(df_future, pred_probs, bitrate_kbps=BITRATE_KBPS, min_setup_s=MIN_SETUP_S, max_selected=MAX_SCHEDULED)
        chosen_ids_milp = [df_future.iloc[i]['pass_id'] for i, chosen in enumerate(chosen_bool) if chosen]
        #print(f'Passes agendados pela política preditiva (MILP): {chosen_ids_milp}')

        greedy_ids_by_start = greedy_schedule_by_start(df_future)
        greedy_ids_by_value = greedy_schedule_by_expected_value(df_future, expected_values, min_setup_s=MIN_SETUP_S)
        satnogs_ids = set(df_future[df_future['max_elevation_deg'] >= 20]['pass_id'].tolist())

        results = {}
        results['predictive'] = evaluate_schedule(df_future, chosen_ids_milp, true_labels, pred_probs)
        results['greedy_start'] = evaluate_schedule(df_future, greedy_ids_by_start, true_labels, pred_probs)
        results['greedy_value'] = evaluate_schedule(df_future, greedy_ids_by_value, true_labels, pred_probs)
        results['satnogs'] = evaluate_schedule(df_future, satnogs_ids, true_labels, pred_probs)
        #print(results)
        print('\n=== Resultados de agendamento (kB) ===')
        for k, v in results.items():
            print(f'{k}: expected_kb={v["expected_kb"]:.1f}, realized_kb={v["realized_kb"]:.1f}')

        baseline = results['greedy_start']['realized_kb'] + 1e-9
        gain_pct = 100.0 * ( baseline - results['predictive']['realized_kb']) / baseline
        print(f'Ganho em throughput (predictive vs greedy_start): {gain_pct:.1f}%')

        # plots de diagnóstico
        plt.figure(figsize=(6,4))
        plt.hist(pred_probs, bins=10)
        plt.title('Distribution of Predicted P(success)')
        plt.xlabel('P(success)')
        plt.ylabel('counts')
        plt.grid(True)
        plt.savefig('distribuicao_p_success.pdf')  # Salva o gráfico em PDF

        plt.show()

        plt.figure(figsize=(6,4))
        plt.scatter(pred_probs, true_labels + rng.normal(scale=0.02, size=true_labels.shape))  # jitter para ver pontos
        plt.xlabel('Predicted P(success)')
        plt.ylabel('True Label (0/1)')
        plt.title('Predicted Probabilities vs True Labels (Future)')
        plt.savefig('pred_probs_vs_true_labels.pdf')  # Salva o gráfico em PD
        plt.show()



        # Verificar qualidade das previsões
        plt.figure(figsize=(10, 6))
        plt.scatter(range(len(true_labels)), true_labels, label='True', alpha=0.7)
        plt.scatter(range(len(pred_probs)), pred_probs, label='Predicted', alpha=0.7)
        plt.legend()
        plt.title('Comparison of True vs Predicted Values')
        plt.savefig('comparacao_reais_vs_preditos.pdf')  # Salva o gráfico em PDF
        plt.grid(True)
        plt.show()
